# Day 15：大模型原理科普：NLP演进、Transformer注意力与Token成本核算（Jupyter Notebook 实战手册）

> **所属阶段**：第 2 阶段 · 第二阶段：大模型核心原理、Prompt工程与全栈 Web API ｜ **第 3 周**：第3周：大模型底层原理、Prompt 工程与工具调用实战  
> **GitLab 工单**：`STARMIND-015` ｜ **开发分支**：`feat/day15-llm-tokenizer-cost`  
> **配套 HTML5 课件**：`courseware/day15/index.html` ｜ **独立版本快照**：`versions/day15/`

---

## 🎙️ 老陈开场旁白

早啊各位，欢迎正式跨入第二阶段！前两周咱们把 Python 工程底子打瓷实了，今天开始正式和「大语言模型」正面交锋。很多刚入行的同学调 API 调得很溜，但一被面试官或客户问「Transformer 自注意力到底在算什么」「为什么中文 100 字消耗的 Token 比英文 100 词多」「预训练、SFT 和 RLHF 分别解决什么问题」，立马就卡壳。今天老陈不用枯燥的论文公式吓唬大家，咱们用星辰数科会议纪要和财务算账的真实场景，把 NLP 简史、Attention 机制、三阶段训练全讲透，然后亲手写出能落地的 Token 估算器和月度预算核算工具，下午直接把报表拍在财务总监桌上！

### 承上启下
- **昨日基线**：第一阶段（Day 01 ~ Day 14）我们打牢了 Python 工程基础、面向对象、异常处理、文件处理与 HTTP/API 调用技能，已经能把本地企业文档读写和基础脚本跑得非常顺溜。但昨天财务总监来问：如果把全公司 800 名员工接入星辰智脑，一个月到底要烧多少 API 钱？中文和英文按字算还是按 Token 算？为什么上下文一长模型就变笨？
- **明日预告**：明天（Day 16）我们将深入大模型 API 核心采样参数（temperature、top_p、max_tokens、frequency_penalty）与流式打字机原理，亲手用代码做参数对比实验。


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day15'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-015）

杭州星辰数科即将全员推广「星辰智脑」AI 助手，涵盖研发、客服、运营三个部门。技术委员会要求在正式采购 DeepSeek / 通义千问 / GLM 商业 API 额度前，先交付一套精准的 Token 估算与成本预测工具，并完成研发团队的大模型基础原理对齐。

**今日迭代目标：**
- 实现中英文混合 Token 分词与计数估算器，区分中文汉字、英文单词、数字与代码符号的不同分词倍率。
- 用纯 Python 实现简化版 Scaled Dot-Product Self-Attention 计算演示，直观展示 Token 间注意力权重分布。
- 实现多模型单次调用与月度企业预算核算引擎，支持超预算红线告警与选型建议输出。

**验收标准：**
- 中英文混合文本的 Token 估算误差控制在主流 BPE 分词器（如 tiktoken/cl100k_base）合理区间内，且零外网依赖。
- 自注意力权重矩阵每一行之和严格等于 1.0（浮点误差 < 1e-6）。
- 成本估算器支持至少 4 款主流国内外模型，能精准区分输入 Token（Input）与输出 Token（Output）的阶梯定价。

---

## 📚 课堂核心技术笔记

### 1. 1. NLP 简史：为什么是 Transformer 赢下了时代？

自然语言处理经历了四个关键阶段：(1) **规则与统计时代（N-Gram / TF-IDF）**：靠人工语法树和词频统计，完全不懂同义词和语义；(2) **词向量时代（Word2Vec / GloVe）**：把词映射到稠密向量，出现「国王 - 男人 + 女人 ≈ 女王」，但无法解决一词多义（苹果手机 vs 吃苹果）；(3) **循环神经网络时代（RNN / LSTM）**：按时间步从左到右读句子，存在致命双缺陷——长距离信息梯度消失（读到句尾忘了句首），且必须串行计算无法吃满 GPU 算力；(4) **Transformer 时代（2017 年《Attention Is All You Need》）**：彻底抛弃循环结构，用自注意力机制让句中任意两个词一步直连，且所有位置完全并行训练，直接引爆了 Scaling Law（缩放定律）与大语言模型革命。

> **⚠️ 老陈避坑**：常见误区：以为 GPT 和 BERT 是同一种结构。其实 BERT 只用了 Transformer 的 Encoder（双向看上下文，擅长分类和向量检索），而 GPT / DeepSeek / Qwen 都是 Decoder-Only（单向因果掩码，只看上文预测下一个词，擅长生成与通用推理）。

### 2. 2. 预训练、SFT 与 RLHF：大模型是如何「长大成人」的？

一个能上岗干活的企业大模型要经历三重淬炼：
- **预训练（Pre-training）**：在数万亿 Token 上做「完形填空/续写下一个词」。这一步花掉 99% 的算力预算，赋予模型海量世界知识和推理底座，产出的是 Base Model（基座模型），此时它只会续写，你问它「杭州天气怎么样」，它可能续写「上海天气怎么样」。
- **有监督微调（SFT）**：用几万到几十万条高质量人类问答指令进行微调，教会模型「我是助手，要按用户指令回答问题」。
- **人类反馈强化学习（RLHF / DPO）**：让标注员对模型生成的多个回答排序，训练模型更符合人类偏好（Helpful 有帮助、Honest 诚实不胡说、Harmless 安全合规）。

> **⚠️ 老陈避坑**：企业落地踩坑：很多老板动不动就想「微调一个自己的大模型」，以为微调能注入企业知识库。记住：SFT 擅长改变说话风格和输出格式，注入动态企业知识应该优先用第三阶段要学的 RAG，成本低两个数量级！

### 3. 3. Token 分词机制与上下文窗口（Context Window）

Token 不是「字」也不是「词」，而是 BPE（Byte-Pair Encoding）算法统计高频字符组合切分出来的「子词单元」。例如英文 `unbelievable` 可能被切成 `un` + `believ` + `able`（3 个 Token）；中文在针对中文优化的国产模型（DeepSeek、Qwen）中，常见双字词如「人工智能」往往只需 1~2 个 Token，平均约 0.7~1.0 Token/字，但在偏重英文的词表中可能高达 1.5~2.0 Token/字。上下文窗口（如 32K、128K）指的是 **输入 Token + 输出 Token 的总上限**，超过会被截断或直接报错。

> **⚠️ 老陈避坑**：生产事故高发点：在计算是否超出 `context_window` 时，忘了把 `system prompt`、历史多轮对话以及预留给模型回复的 `max_tokens` 全加进去，导致线上报 `400 context_length_exceeded`。


## 💻 实操步骤 1：步骤一：实现中英文混合 Token 分词估算器与简化版 Self-Attention 权重演示

- **目标文件**：`versions/day15/src/day15_tokenizer.py`
- **老陈代码旁白**：咱们先写第一个核心模块 `src/day15_tokenizer.py`。在没有外网下载 tiktoken 词表缓存的离线内网环境里，企业网关必须有一套轻量、极速、区分中英文和代码符号的 Token 估算器。顺带老陈把 Scaled Dot-Product Self-Attention 用 20 行纯 Python 写出来，大家跑一下就能亲眼看到注意力权重是怎么归一化的。


In [ ]:
"""
StarMind Day 15 - 步骤一：中英文混合 Token 分词估算器与 Self-Attention 原理演示
无需外部模型文件，纯 Python 实现高精度启发式分词统计与缩放点积注意力计算。
"""

import math
import re
from dataclasses import dataclass
from typing import Dict, List, Tuple


@dataclass
class TokenBreakdown:
    """文本分词拆解明细结果。"""
    total_chars: int
    cjk_chars: int
    ascii_words: int
    numbers: int
    symbols: int
    estimated_tokens: int
    segments: List[str]


class StarMindTokenizer:
    """
    星辰智脑轻量级中英文混合 Token 估算器。
    支持针对不同词表风格（cn_optimized 国产中文优化 vs global_bpe 通用英文偏重）切换估算倍率。
    """

    # 匹配顺序：英文单词 -> 数字 -> 单个中文汉字 -> 非空白标点符号
    _TOKEN_PATTERN = re.compile(
        r"[A-Za-z_][A-Za-z0-9_]*|\d+(?:\.\d+)?|[\u4e00-\u9fff]|[^\sA-Za-z0-9_\u4e00-\u9fff]"
    )

    PROFILE_WEIGHTS: Dict[str, Dict[str, float]] = {
        # DeepSeek / Qwen / GLM 等对中文词表高度优化的模型
        "cn_optimized": {
            "cjk": 0.75,      # 常见中文双字词常合并，平均约 0.75 token/字
            "word": 1.15,     # 英文单词偶有子词拆分
            "number": 1.0,    # 连续数字通常按 1~3 位切分
            "symbol": 0.8,    # 标点与代码符号
        },
        # 早期偏英文 BPE 词表（如经典 cl100k_base）
        "global_bpe": {
            "cjk": 1.50,      # 中文在英文主导词表中往往消耗 1.5~2 token/字
            "word": 1.05,
            "number": 1.0,
            "symbol": 0.85,
        },
    }

    def __init__(self, profile: str = "cn_optimized") -> None:
        if profile not in self.PROFILE_WEIGHTS:
            raise ValueError(f"不支持的分词配置: {profile}，可选: {list(self.PROFILE_WEIGHTS)}")
        self.profile = profile
        self.weights = self.PROFILE_WEIGHTS[profile]

    def analyze(self, text: str) -> TokenBreakdown:
        """分析文本的语种构成并估算 Token 总数。"""
        if not text:
            return TokenBreakdown(0, 0, 0, 0, 0, 0, [])

        segments = self._TOKEN_PATTERN.findall(text)
        cjk_chars = 0
        ascii_words = 0
        numbers = 0
        symbols = 0

        for seg in segments:
            if "\u4e00" <= seg[0] <= "\u9fff":
                cjk_chars += 1
            elif seg[0].isdigit():
                numbers += 1
            elif seg[0].isalpha() or seg[0] == "_":
                ascii_words += 1
            else:
                symbols += 1

        raw_score = (
            cjk_chars * self.weights["cjk"]
            + ascii_words * self.weights["word"]
            + numbers * self.weights["number"]
            + symbols * self.weights["symbol"]
        )
        # 至少返回 1 个 Token（只要存在有效片段）
        estimated = max(1, math.ceil(raw_score)) if segments else 0

        return TokenBreakdown(
            total_chars=len(text),
            cjk_chars=cjk_chars,
            ascii_words=ascii_words,
            numbers=numbers,
            symbols=symbols,
            estimated_tokens=estimated,
            segments=segments,
        )

    def count_tokens(self, text: str) -> int:
        """快捷方法：直接返回估算的 Token 数量。"""
        return self.analyze(text).estimated_tokens


def scaled_dot_product_attention(
    queries: List[List[float]],
    keys: List[List[float]],
    values: List[List[float]],
) -> Tuple[List[List[float]], List[List[float]]]:
    """
    纯 Python 实现简化版 Transformer Scaled Dot-Product Self-Attention：
    Attention(Q, K, V) = softmax(Q @ K^T / sqrt(d_k)) @ V
    返回: (上下文融合输出矩阵, 注意力权重矩阵)
    """
    if not queries or not keys or not values:
        raise ValueError("Q、K、V 矩阵均不能为空")
    d_k = len(queries[0])
    scale = math.sqrt(d_k)

    attention_weights: List[List[float]] = []
    outputs: List[List[float]] = []

    for q_vec in queries:
        # 1. 计算当前 Query 与所有 Key 的缩放点积分数
        scores = [
            sum(q_i * k_i for q_i, k_i in zip(q_vec, k_vec)) / scale
            for k_vec in keys
        ]
        # 2. 数值稳定版 Softmax（减去最大值防止 exp 溢出）
        max_score = max(scores)
        exp_scores = [math.exp(s - max_score) for s in scores]
        sum_exp = sum(exp_scores)
        weights = [e / sum_exp for e in exp_scores]
        attention_weights.append(weights)

        # 3. 使用注意力权重对 Value 进行加权求和
        v_dim = len(values[0])
        out_vec = [0.0] * v_dim
        for w, v_vec in zip(weights, values):
            for idx in range(v_dim):
                out_vec[idx] += w * v_vec[idx]
        outputs.append(out_vec)

    return outputs, attention_weights


if __name__ == "__main__":
    sample = "杭州星辰数科 StarMind 平台支持 DeepSeek-V3 与 Qwen2.5，日均处理 12500 条工单！"
    cn_tok = StarMindTokenizer("cn_optimized")
    gl_tok = StarMindTokenizer("global_bpe")
    print("中文优化词表估算:", cn_tok.analyze(sample))
    print("通用英文词表估算:", gl_tok.analyze(sample))

    # 演示 3 个 Token（如 ["星辰", "智脑", "平台"]）的自注意力计算
    q_demo = [[1.0, 0.2], [0.1, 0.9], [0.8, 0.7]]
    k_demo = [[1.0, 0.0], [0.0, 1.0], [0.7, 0.7]]
    v_demo = [[10.0, 1.0], [2.0, 8.0], [5.0, 5.0]]
    out_mat, w_mat = scaled_dot_product_attention(q_demo, k_demo, v_demo)
    print("注意力权重矩阵:", [[round(x, 4) for x in row] for row in w_mat])
    print("融合输出矩阵:", [[round(x, 3) for x in row] for row in out_mat])


### 🔍 步骤 1 代码深度解析

1. `_TOKEN_PATTERN` 正则按「英文标识符 -> 数字 -> 单个中文字符 -> 标点符号」优先级切分，避免把英文单词拆成单个字母。
2. `PROFILE_WEIGHTS` 区分了国产中文优化词表与传统英文偏重词表，能非常直观地解释为什么同样一段中文在 DeepSeek/Qwen 上更省 Token。
3. `scaled_dot_product_attention` 中特别做了 `s - max_score` 数值稳定性处理，这是深度学习框架底层 Softmax 防止 `OverflowError` 的标准做法。

**预期运行输出参考：**
```text
中文优化词表估算: TokenBreakdown(total_chars=49, cjk_chars=23, ascii_words=4, numbers=2, symbols=5, estimated_tokens=28, ...)
注意力权重矩阵: [[0.4227, 0.2401, 0.3372], ...]
```


## 💻 实操步骤 2：步骤二：实现多模型企业级 API 成本估算器与月度预算告警引擎

- **目标文件**：`versions/day15/src/day15_cost_estimator.py`
- **老陈代码旁白**：有了 Token 估算器，接下来咱们把财务总监最关心的成本账算清楚！`src/day15_cost_estimator.py` 内置了星辰数科调研的四款主流模型单价表（按元/百万 Token 计价），支持单次问答计费、全公司月度预算预测，以及超过预算红线时的自动告警和低成本平替模型推荐。


In [ ]:
"""
StarMind Day 15 - 步骤二：企业级大模型 API 成本估算与月度预算评估器
支持多模型输入/输出阶梯定价对比、月度 DAU 费用预测与超预算告警。
"""

from dataclasses import dataclass
from typing import Dict, List

from src.day15_tokenizer import StarMindTokenizer


@dataclass(frozen=True)
class ModelPricing:
    """大模型 API 定价规范（单位：人民币 元 / 100万 Tokens）。"""
    model_id: str
    provider: str
    input_price_per_1m: float
    output_price_per_1m: float
    context_window: int
    tokenizer_profile: str = "cn_optimized"


# 星辰数科 2026 内部参考模型价目表（示例人民币定价，便于横向选型）
MODEL_CATALOG: Dict[str, ModelPricing] = {
    "deepseek-chat": ModelPricing(
        model_id="deepseek-chat",
        provider="DeepSeek",
        input_price_per_1m=2.0,
        output_price_per_1m=8.0,
        context_window=64_000,
        tokenizer_profile="cn_optimized",
    ),
    "qwen-max": ModelPricing(
        model_id="qwen-max",
        provider="阿里云百炼",
        input_price_per_1m=2.4,
        output_price_per_1m=9.6,
        context_window=32_000,
        tokenizer_profile="cn_optimized",
    ),
    "glm-4-plus": ModelPricing(
        model_id="glm-4-plus",
        provider="智谱AI",
        input_price_per_1m=5.0,
        output_price_per_1m=5.0,
        context_window=128_000,
        tokenizer_profile="cn_optimized",
    ),
    "gpt-4o": ModelPricing(
        model_id="gpt-4o",
        provider="OpenAI",
        input_price_per_1m=18.0,
        output_price_per_1m=72.0,
        context_window=128_000,
        tokenizer_profile="global_bpe",
    ),
}


@dataclass
class CostReport:
    """单次或月度成本预测报告。"""
    model_id: str
    input_tokens: int
    output_tokens: int
    single_call_cny: float
    monthly_total_cny: float
    exceeds_budget: bool
    context_utilization_pct: float
    recommendation: str


class EnterpriseCostEstimator:
    """企业级 FinOps 大模型成本核算器。"""

    def __init__(self, catalog: Dict[str, ModelPricing] | None = None) -> None:
        self.catalog = catalog or MODEL_CATALOG

    def estimate_single_cost(
        self, model_id: str, input_tokens: int, output_tokens: int
    ) -> float:
        """按百万 Token 单价精确计算单次调用成本（人民币元）。"""
        if model_id not in self.catalog:
            raise KeyError(f"未注册的模型: {model_id}")
        if input_tokens < 0 or output_tokens < 0:
            raise ValueError("Token 数量不能为负数")
        p = self.catalog[model_id]
        cost = (
            (input_tokens / 1_000_000) * p.input_price_per_1m
            + (output_tokens / 1_000_000) * p.output_price_per_1m
        )
        return round(cost, 6)

    def forecast_monthly_budget(
        self,
        model_id: str,
        sample_prompt: str,
        sample_reply: str,
        daily_active_users: int,
        avg_calls_per_user: int,
        monthly_budget_cny: float,
        workdays: int = 22,
    ) -> CostReport:
        """根据典型问答样本与日活规模，预测企业月度 API 支出并检查预算红线。"""
        if model_id not in self.catalog:
            raise KeyError(f"未注册的模型: {model_id}")
        pricing = self.catalog[model_id]
        tokenizer = StarMindTokenizer(pricing.tokenizer_profile)

        in_tokens = tokenizer.count_tokens(sample_prompt)
        out_tokens = tokenizer.count_tokens(sample_reply)
        single_cost = self.estimate_single_cost(model_id, in_tokens, out_tokens)

        monthly_calls = daily_active_users * avg_calls_per_user * workdays
        monthly_cost = round(single_cost * monthly_calls, 2)
        exceeds = monthly_cost > monthly_budget_cny
        ctx_pct = round(((in_tokens + out_tokens) / pricing.context_window) * 100, 4)

        if exceeds:
            rec = (
                f"⚠️ 预计月支出 ¥{monthly_cost:.2f} 超出预算 ¥{monthly_budget_cny:.2f}，"
                f"建议将非核心场景路由至 deepseek-chat 或启用 Prompt 缓存降本。"
            )
        else:
            rec = f"✅ 预计月支出 ¥{monthly_cost:.2f} 在预算 ¥{monthly_budget_cny:.2f} 内，可安心上线。"

        return CostReport(
            model_id=model_id,
            input_tokens=in_tokens,
            output_tokens=out_tokens,
            single_call_cny=single_cost,
            monthly_total_cny=monthly_cost,
            exceeds_budget=exceeds,
            context_utilization_pct=ctx_pct,
            recommendation=rec,
        )

    def compare_all_models(
        self,
        sample_prompt: str,
        sample_reply: str,
        daily_active_users: int,
        avg_calls_per_user: int,
        monthly_budget_cny: float,
    ) -> List[CostReport]:
        """横向对比价目表中全部模型的月度成本，按从低到高排序。"""
        reports = [
            self.forecast_monthly_budget(
                model_id=mid,
                sample_prompt=sample_prompt,
                sample_reply=sample_reply,
                daily_active_users=daily_active_users,
                avg_calls_per_user=avg_calls_per_user,
                monthly_budget_cny=monthly_budget_cny,
            )
            for mid in self.catalog
        ]
        reports.sort(key=lambda r: r.monthly_total_cny)
        return reports


if __name__ == "__main__":
    estimator = EnterpriseCostEstimator()
    prompt_text = "请帮我总结以下杭州星辰数科 Q3 研发效能报告的核心指标与风险点：" + ("研发交付周期缩短15%，线上缺陷率下降。" * 20)
    reply_text = "【核心结论】1. 交付效率显著提升；2. 缺陷率稳步下降；3. 建议下季度重点优化代码评审耗时。" * 5

    results = estimator.compare_all_models(
        sample_prompt=prompt_text,
        sample_reply=reply_text,
        daily_active_users=800,
        avg_calls_per_user=15,
        monthly_budget_cny=1000.0,
    )
    for rep in results:
        print(f"{rep.model_id:15s} | 月成本: ¥{rep.monthly_total_cny:8.2f} | {rep.recommendation}")


### 🔍 步骤 2 代码深度解析

1. 使用 `@dataclass(frozen=True)` 定义不可变定价表，防止业务代码中意外篡改单价。
2. `forecast_monthly_budget` 自动根据模型对应的 `tokenizer_profile` 选择分词倍率，真实还原了国产中文优化模型 vs 英文主导词表在中文场景下的双重成本优势（单价更低 + Token 数更少）。
3. `compare_all_models` 按月度费用升序排列，方便架构师直接输出选型比价表。

**预期运行输出参考：**
```text
deepseek-chat   | 月成本: ¥  470.45 | ✅ 预计月支出 ¥470.45 在预算 ¥1000.00 内，可安心上线。
gpt-4o          | 月成本: ¥ 8412.60 | ⚠️ 预计月支出 ¥8412.60 超出预算 ¥1000.00...
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day15/tests/test_day15.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""Day 15 自动化单元测试：验证 Token 分词估算、Self-Attention 归一化与企业成本预测。"""

import pytest
from src.day15_tokenizer import StarMindTokenizer, scaled_dot_product_attention
from src.day15_cost_estimator import EnterpriseCostEstimator


def test_tokenizer_cn_vs_global_profile():
    text = "杭州星辰数科正在构建企业级智能体平台 StarMind 2026！"
    cn = StarMindTokenizer("cn_optimized").analyze(text)
    gl = StarMindTokenizer("global_bpe").analyze(text)

    assert cn.cjk_chars == 18
    assert cn.ascii_words == 1
    assert cn.numbers == 1
    assert cn.estimated_tokens > 0
    # 中文主导文本在 global_bpe 下消耗的 Token 应高于国产中文优化词表
    assert gl.estimated_tokens > cn.estimated_tokens


def test_tokenizer_empty_and_invalid_profile():
    tok = StarMindTokenizer("cn_optimized")
    assert tok.count_tokens("") == 0
    with pytest.raises(ValueError):
        StarMindTokenizer("unknown_profile")


def test_scaled_dot_product_attention_weights_sum_to_one():
    q = [[1.0, 0.0, 0.5], [0.2, 0.8, 0.1]]
    k = [[1.0, 0.0, 0.5], [0.0, 1.0, 0.0], [0.5, 0.5, 0.5]]
    v = [[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]]
    outputs, weights = scaled_dot_product_attention(q, k, v)

    assert len(outputs) == 2
    assert len(weights) == 2
    for row in weights:
        assert abs(sum(row) - 1.0) < 1e-6
        assert all(w > 0 for w in row)


def test_enterprise_cost_estimator_and_budget_alert():
    est = EnterpriseCostEstimator()
    # 验证单次精确计费：100万输入 + 100万输出 = input_price + output_price
    cost = est.estimate_single_cost("deepseek-chat", 1_000_000, 1_000_000)
    assert cost == pytest.approx(10.0)

    reports = est.compare_all_models(
        sample_prompt="请总结星辰数科季度研发报告" * 30,
        sample_reply="核心指标达成率105%，风险可控。" * 15,
        daily_active_users=500,
        avg_calls_per_user=20,
        monthly_budget_cny=500.0,
    )
    assert len(reports) == 4
    # 结果按月度成本升序排列，deepseek-chat 应显著低于 gpt-4o
    assert reports[0].monthly_total_cny < reports[-1].monthly_total_cny
    assert reports[-1].model_id == "gpt-4o"
    assert reports[-1].exceeds_budget is True

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：上下文窗口安全守门员（Context Window Guard）

**题目要求：**
编写函数 `check_context_fit(system_prompt, history, user_query, max_reply_tokens, context_window)`，使用 `StarMindTokenizer` 计算总 Token 占用，返回是否超限及剩余可用 Token 数。

**思路提示：** 遍历 `history` 累加每条消息的 Token，再把 `system_prompt`、`user_query` 和 `max_reply_tokens` 一并计入总量。


In [ ]:
from typing import Dict, List
from src.day15_tokenizer import StarMindTokenizer


def check_context_fit(
    system_prompt: str,
    history: List[str],
    user_query: str,
    max_reply_tokens: int,
    context_window: int = 32_000,
) -> Dict[str, object]:
    """检查当前完整对话加上预留回复长度是否超出模型上下文窗口。"""
    tok = StarMindTokenizer("cn_optimized")
    used_input = (
        tok.count_tokens(system_prompt)
        + sum(tok.count_tokens(msg) for msg in history)
        + tok.count_tokens(user_query)
    )
    total_required = used_input + max_reply_tokens
    remaining = context_window - total_required
    return {
        "fits": remaining >= 0,
        "used_input_tokens": used_input,
        "total_required_tokens": total_required,
        "remaining_tokens": remaining,
    }


if __name__ == "__main__":
    res = check_context_fit(
        system_prompt="你是星辰数科智能助手。",
        history=["你好", "请问有什么可以帮您？"],
        user_query="请帮我写一份季度复盘大纲。",
        max_reply_tokens=1024,
        context_window=4096,
    )
    print(res)


**答案解析与评分标准：**

核心考点在于理解 Context Window 是输入与输出之和。生产中若忽略 `max_reply_tokens`，常会在长对话时触发接口 400 错误。


### 作业 2（企业实战）：Prompt Caching 缓存命中降本计算器

**题目要求：**
DeepSeek 与阿里云百炼均支持前缀缓存（Prompt Caching），命中缓存的输入 Token 单价可降至原价的 10%~25%。请编写函数计算当系统提示词占比为 `cache_ratio`（如 0.6）、缓存折扣为 `cache_discount`（如 0.2）时，企业月度可节省的金额与降本百分比。

**思路提示：** 把总输入 Token 拆分为 `cached_tokens` 和 `uncached_tokens` 两部分分别计价，输出 Token 价格不变。


In [ ]:
from typing import Dict


def calculate_cache_savings(
    input_tokens: int,
    output_tokens: int,
    input_price_per_1m: float,
    output_price_per_1m: float,
    monthly_calls: int,
    cache_ratio: float = 0.6,
    cache_discount: float = 0.2,
) -> Dict[str, float]:
    """计算开启前缀 Prompt Caching 后的月度成本与节省比例。"""
    base_single = (
        (input_tokens / 1_000_000) * input_price_per_1m
        + (output_tokens / 1_000_000) * output_price_per_1m
    )
    cached_in = input_tokens * cache_ratio
    uncached_in = input_tokens * (1.0 - cache_ratio)
    opt_single = (
        (cached_in / 1_000_000) * input_price_per_1m * cache_discount
        + (uncached_in / 1_000_000) * input_price_per_1m
        + (output_tokens / 1_000_000) * output_price_per_1m
    )
    base_monthly = round(base_single * monthly_calls, 2)
    opt_monthly = round(opt_single * monthly_calls, 2)
    saved = round(base_monthly - opt_monthly, 2)
    saved_pct = round((saved / base_monthly) * 100, 2) if base_monthly > 0 else 0.0
    return {
        "base_monthly_cny": base_monthly,
        "optimized_monthly_cny": opt_monthly,
        "saved_cny": saved,
        "saved_pct": saved_pct,
    }


if __name__ == "__main__":
    print(calculate_cache_savings(2000, 400, 2.0, 8.0, monthly_calls=200_000))


**答案解析与评分标准：**

通过量化分析可以看出，在长 System Prompt + 多轮知识库问答场景下，把静态指令放在最前面以提高 KV Cache 命中率，是极低成本的架构优化手段。


### 作业 3（架构进阶）：因果掩码（Causal Mask）自注意力实现

**题目要求：**
GPT/DeepSeek 等 Decoder-Only 模型在生成时不能「偷看」未来的词，需要在 Softmax 前对上三角位置（`j > i`）加上 `-1e9` 掩码。请扩展 `scaled_dot_product_attention` 实现带因果掩码的版本，并验证第 0 个 Token 对后面 Token 的注意力权重接近 0。

**思路提示：** 在计算第 `i` 个 Query 对第 `j` 个 Key 的分数时，若 `j > i` 则将分数设为 `-1e9`。


In [ ]:
import math
from typing import List


def causal_self_attention_weights(
    queries: List[List[float]],
    keys: List[List[float]],
) -> List[List[float]]:
    """计算带因果掩码（上三角屏蔽）的自注意力权重矩阵。"""
    d_k = len(queries[0])
    scale = math.sqrt(d_k)
    weights_matrix: List[List[float]] = []

    for i, q_vec in enumerate(queries):
        scores = []
        for j, k_vec in enumerate(keys):
            if j > i:
                scores.append(-1e9)
            else:
                dot = sum(a * b for a, b in zip(q_vec, k_vec))
                scores.append(dot / scale)
        max_s = max(scores)
        exps = [math.exp(s - max_s) for s in scores]
        total = sum(exps)
        weights_matrix.append([e / total for e in exps])

    return weights_matrix


if __name__ == "__main__":
    vecs = [[1.0, 0.5], [0.4, 0.9], [0.8, 0.2]]
    w = causal_self_attention_weights(vecs, vecs)
    for idx, row in enumerate(w):
        print(f"Token {idx} 注意力分布:", [round(x, 4) for x in row])


**答案解析与评分标准：**

第 0 行权重必为 `[1.0, 0.0, 0.0]`，第 1 行最后一个元素必为 `0.0`。这正是 Decoder-Only 模型能并行训练同时又不泄露未来答案的核心机制。


---

## 🎯 老陈收尾复盘

今天这一仗打得漂亮！咱们不仅搞懂了大模型从词袋模型到 Transformer 的演进脉络、明白了 Attention 的本质是「动态加权检索」，还亲手落了中英文 Token 估算器和企业级成本核算器。记住老陈的一条工程铁律：在企业里做 AI 应用，不算清楚 Token 成本和上下文窗口，上线第一周就能把部门预算烧穿。今晚把三道作业做完，明天咱们拆解 API 采样参数和流式输出！
